# B.6 — SpatialPeeler benchmark evaluation: 3×3×3 parameter grid

Runs SpatialPeeler on all 27 benchmark conditions generated by `B.5.benchmark_nmfGenes_grid.ipynb`.

**Conditions:** 3 perturb_fracs × 3 lambdas × 3 top_genes = 27 case files in `generated_benchmark_data_final/`.

**Metric:** spot-based in-circle AUROC — for each condition, the top factor chosen by  
SpatialPeeler (highest positive logistic regression coefficient) is evaluated against  
the ground-truth `obs['in_circle']` column on case beads only.

In [ ]:
import sys
import warnings
from itertools import product
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import statsmodels.api as sm
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 0

ROOT    = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
sys.path.insert(0, str(ROOT))

RAND_SEED  = 28
N_FACTORS  = 15
np.random.seed(RAND_SEED)

DATA_DIR  = ROOT / 'benchmark' / 'generated_benchmark_data'
FINAL_DIR = ROOT / 'benchmark' / 'generated_benchmark_data_final'

print('ROOT:', ROOT)
print('Case files:', len(list(FINAL_DIR.glob('*.h5ad'))))

## 1. Load control (shared across all conditions)

In [ ]:
adata_ctrl = ad.read_h5ad(DATA_DIR / 'adata06_top.h5ad')
adata_ctrl.obs['sample_id'] = 'ctrl'
adata_ctrl.obs['status']    = 0
adata_ctrl.obs['Condition'] = 'Control'
if 'in_circle' not in adata_ctrl.obs.columns:
    adata_ctrl.obs['in_circle'] = False
adata_ctrl.obs['in_circle'] = adata_ctrl.obs['in_circle'].astype(bool)

print('Control:', adata_ctrl.shape)

## 2. Define the parameter grid

In [ ]:
PERTURB_FRACS  = [0.30, 0.50, 0.70]
FIXED_LAMS     = [0.5,  0.3,  0.15]
TOP_GENES_LIST = [1, 5, 10]

grid = list(product(PERTURB_FRACS, FIXED_LAMS, TOP_GENES_LIST))
print(f'{len(grid)} conditions')

## 3. Helper functions

In [ ]:
def preprocess(adata_ctrl, adata_case):
    """Concat, filter, normalise, HVG, scale. Returns combined adata."""
    adata = ad.concat(
        [adata_ctrl, adata_case],
        join='inner', merge='first',
        label='sample_id', keys=['ctrl', 'case'],
        index_unique='-'
    )
    for col in ['status', 'Condition', 'in_circle']:
        adata.obs[col] = np.concatenate([
            adata_ctrl.obs[col].values,
            adata_case.obs[col].values
        ])
    adata.obs['status']    = adata.obs['status'].astype(int)
    adata.obs['in_circle'] = adata.obs['in_circle'].astype(bool)

    adata.layers['counts'] = adata.X.copy()

    # gene filter
    min_cells = max(1, adata.n_obs // 500)
    n_expr    = np.array((adata.X > 0).sum(axis=0)).flatten()
    adata     = adata[:, n_expr >= min_cells].copy()

    # lognorm + HVG
    adata.layers['lognorm'] = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4, layer='lognorm')
    sc.pp.log1p(adata, layer='lognorm')
    sc.pp.highly_variable_genes(
        adata, n_top_genes=2000,
        batch_key='sample_id', flavor='seurat',
        layer='lognorm', subset=False
    )
    adata = adata[:, adata.var['highly_variable']].copy()

    sc.pp.scale(adata, zero_center=False)
    return adata


def run_nmf(adata, n_factors=N_FACTORS, rand_seed=RAND_SEED):
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = X.astype(np.float32)
    model = NMF(n_components=n_factors, init='nndsvda',
                random_state=rand_seed, max_iter=1000, solver='cd')
    W = model.fit_transform(X)
    adata.obsm['X_nmf'] = W
    return W


def run_spatialpeeler(adata, n_factors=N_FACTORS):
    """Logistic regression for each factor. Returns list of result dicts."""
    X  = adata.obsm['X_nmf']
    y  = adata.obs['status'].values
    results = []
    for i in range(n_factors):
        Xi    = X[:, i].reshape(-1, 1)
        X_int = sm.add_constant(Xi)
        try:
            fit = sm.Logit(y, X_int).fit(disp=False)
            p_hat = fit.predict(X_int)
            coef  = float(fit.params[1])
            pval  = float(fit.pvalues[1])
        except Exception:
            p_hat = np.full(len(y), np.nan)
            coef  = 0.0
            pval  = 1.0
        results.append({'factor_index': i, 'coef': coef, 'pval': pval, 'p_hat': p_hat})
    return results


def incircle_auroc(results, adata):
    """For each factor, compute AUROC of p_hat vs in_circle on case beads."""
    case_mask = adata.obs['status'].values == 1
    gt        = adata.obs['in_circle'].values[case_mask].astype(int)
    aucs = {}
    for r in results:
        ph_case = r['p_hat'][case_mask]
        if np.isnan(ph_case).all() or len(np.unique(gt)) < 2:
            aucs[r['factor_index']] = np.nan
        else:
            try:
                aucs[r['factor_index']] = roc_auc_score(gt, ph_case)
            except Exception:
                aucs[r['factor_index']] = np.nan
    return aucs


print('Helper functions ready.')

## 4. Run all 27 conditions

For each condition:
1. Load case h5ad
2. Preprocess (concat with ctrl, HVG, scale)
3. NMF (k=15)
4. SpatialPeeler logistic regression
5. Pick top factor (highest positive coefficient)
6. Record in-circle AUROC for that factor

In [ ]:
rows = []

for i, (pf, lam, tg) in enumerate(grid):
    tag  = f'frac{pf:.0%}_lam{lam}_top{tg}genes'
    path = FINAL_DIR / f'adata06_bot_case_nmfGenes_{tag}.h5ad'

    print(f'[{i+1:02d}/27] {tag} ...', end=' ', flush=True)

    # --- load case ---
    adata_case = ad.read_h5ad(path)
    adata_case.obs['sample_id'] = 'case'
    adata_case.obs['status']    = 1
    adata_case.obs['Condition'] = 'Case'
    adata_case.obs['in_circle'] = adata_case.obs['in_circle'].astype(bool)

    n_in_circle = adata_case.obs['in_circle'].sum()

    # --- preprocess ---
    adata = preprocess(adata_ctrl, adata_case)

    # --- NMF ---
    run_nmf(adata)

    # --- SpatialPeeler ---
    results = run_spatialpeeler(adata)

    # --- in-circle AUROC per factor ---
    aucs = incircle_auroc(results, adata)

    # --- pick top factor (highest positive coefficient) ---
    top_r   = max(results, key=lambda r: r['coef'])
    top_fi  = top_r['factor_index']
    top_auc = aucs[top_fi]
    best_auc = max(aucs.values())  # oracle: best across all factors

    print(f'top_factor=F{top_fi+1}  coef={top_r["coef"]:+.2f}  '
          f'top_auc={top_auc:.3f}  best_auc={best_auc:.3f}')

    rows.append({
        'perturb_frac': pf,
        'fixed_lam':    lam,
        'top_genes':    tg,
        'tag':          tag,
        'n_in_circle':  int(n_in_circle),
        'top_factor':   top_fi + 1,
        'top_coef':     top_r['coef'],
        'top_pval':     top_r['pval'],
        'top_auc':      top_auc,   # AUC of the factor SpatialPeeler picks
        'best_auc':     best_auc,  # oracle: best AUC across all factors
    })

results_df = pd.DataFrame(rows)
print('\nDone.')
print(results_df[['tag', 'top_factor', 'top_coef', 'top_auc', 'best_auc']].to_string(index=False))

## 5. Results summary

In [ ]:
print('top_auc (SpatialPeeler top factor)')
print(results_df.groupby(['perturb_frac', 'fixed_lam', 'top_genes'])['top_auc']
      .mean().unstack('top_genes').round(3).to_string())
print()
print('best_auc (oracle: best across all factors)')
print(results_df.groupby(['perturb_frac', 'fixed_lam', 'top_genes'])['best_auc']
      .mean().unstack('top_genes').round(3).to_string())

## 6. Visualise: in-circle AUROC heatmaps across the parameter grid

In [ ]:
def plot_auc_grid(results_df, auc_col, title):
    fig, axes = plt.subplots(
        1, len(TOP_GENES_LIST),
        figsize=(4 * len(TOP_GENES_LIST), 3.5),
        sharey=True
    )
    vmin, vmax = 0.4, 1.0

    for ax, tg in zip(axes, TOP_GENES_LIST):
        sub  = results_df[results_df['top_genes'] == tg]
        mat  = sub.pivot(index='fixed_lam', columns='perturb_frac', values=auc_col)
        # sort so strongest perturbation is top-left
        mat  = mat.sort_index(ascending=False)

        im = ax.imshow(mat.values, cmap='YlOrRd', vmin=vmin, vmax=vmax, aspect='auto')

        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels([f'{c:.0%}' for c in mat.columns], fontsize=9)
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels([str(r) for r in mat.index], fontsize=9)
        ax.set_xlabel('perturb_frac', fontsize=9)
        if ax is axes[0]:
            ax.set_ylabel('fixed_lam', fontsize=9)
        ax.set_title(f'top_genes={tg}', fontsize=10)

        # annotate cells
        for r_i, row in enumerate(mat.values):
            for c_i, val in enumerate(row):
                ax.text(c_i, r_i, f'{val:.2f}', ha='center', va='center',
                        fontsize=8, color='black')

    fig.colorbar(im, ax=axes, shrink=0.8, label='AUROC')
    fig.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


plot_auc_grid(results_df, 'top_auc',
              'In-circle AUROC — SpatialPeeler top factor (by coefficient)')
plot_auc_grid(results_df, 'best_auc',
              'In-circle AUROC — oracle best factor across all 15')

In [ ]:
# Marginal summaries: mean AUROC across the grid
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, col, label in zip(
    axes,
    ['perturb_frac', 'fixed_lam', 'top_genes'],
    ['Perturb fraction', 'Lambda', 'Top genes per factor']
):
    mn = results_df.groupby(col)['top_auc'].mean()
    ax.bar([str(x) for x in mn.index], mn.values, color='steelblue')
    ax.axhline(0.5, color='black', linestyle='--', linewidth=0.8, label='random')
    ax.set_ylim(0.4, 1.0)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Mean AUROC', fontsize=10)
    ax.set_title(f'AUROC by {label}', fontsize=11)
    ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# Gap: how much does SpatialPeeler lose vs the oracle?
results_df['auc_gap'] = results_df['best_auc'] - results_df['top_auc']

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(results_df)), results_df.sort_values('auc_gap', ascending=False)['auc_gap'],
       color='coral')
ax.set_xlabel('Condition (sorted by gap)')
ax.set_ylabel('AUC gap  (oracle − top-factor)')
ax.set_title('How often does SpatialPeeler miss the best factor?')
ax.axhline(0, color='black', linewidth=0.6)
plt.tight_layout()
plt.show()

print(f'Mean gap:   {results_df["auc_gap"].mean():.3f}')
print(f'Cases where top_auc == best_auc: '
      f'{(results_df["auc_gap"] < 1e-6).sum()} / {len(results_df)}')